# Notebook 02: Preprocessing

Image preprocessing, augmentation, and tabular feature engineering pipelines.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

BASE_DIR = Path("..")
META_PATH = BASE_DIR / "data" / "metadata.csv"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
CLASSES = ["Normal", "Mild", "Moderate", "Severe"]

print(f"TensorFlow version: {tf.__version__}")


## 2. Load Metadata

In [ ]:
df = pd.read_csv(META_PATH)
print(f"Total samples: {len(df)}")
print(df["diagnosis"].value_counts())


## 3. Image Loading and Normalisation

In [ ]:
def load_image(image_path: str, img_size: tuple = IMG_SIZE) -> np.ndarray:
    """Load a conjunctiva image, resize to img_size, and normalise pixels to [0, 1].

    Args:
        image_path: Absolute or relative path to the image file.
        img_size: Target (width, height) tuple.

    Returns:
        Normalised float32 numpy array of shape (height, width, 3).
    """
    from PIL import Image as PILImage
    img = PILImage.open(image_path).convert("RGB")
    img = img.resize(img_size)
    img_array = np.array(img, dtype=np.float32) / 255.0
    return img_array


# Quick sanity check
sample_path = str(BASE_DIR / df["image_path"].iloc[0])
sample = load_image(sample_path)
print(f"Loaded image shape: {sample.shape}")
print(f"Pixel range: [{sample.min():.2f}, {sample.max():.2f}]")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(sample)
ax.set_title(f"Sample: {df['diagnosis'].iloc[0]}")
ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Data Augmentation Pipeline

In [ ]:
# TensorFlow augmentation layers applied only during training
augmentation_pipeline = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2),
], name="augmentation")


def augment_image(image, label):
    """Apply augmentation to a single image tensor."""
    image = augmentation_pipeline(image, training=True)
    return image, label


# Visualise augmented examples
sample_tensor = tf.expand_dims(tf.constant(sample), axis=0)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(sample)
axes[0].set_title("Original")
axes[0].axis("off")
for i in range(1, 5):
    aug = augmentation_pipeline(sample_tensor, training=True)[0].numpy()
    aug = np.clip(aug, 0.0, 1.0)
    axes[i].imshow(aug)
    axes[i].set_title(f"Aug {i}")
    axes[i].axis("off")
plt.suptitle("Data Augmentation Examples", fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Tabular Feature Engineering

In [ ]:
# 5a. Encode gender
le_gender = LabelEncoder()
df["gender_enc"] = le_gender.fit_transform(df["gender"])
print("Gender encoding:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))

# 5b. Scale age
scaler_age = StandardScaler()
df["age_scaled"] = scaler_age.fit_transform(df[["age"]])

# 5c. Encode target label
le_label = LabelEncoder()
le_label.fit(CLASSES)
df["label"] = le_label.transform(df["diagnosis"])
print("Label encoding:", dict(zip(le_label.classes_, le_label.transform(le_label.classes_))))

print(df[["age", "age_scaled", "gender", "gender_enc", "diagnosis", "label"]].head())


## 6. Stratified Train / Validation / Test Split

In [ ]:
# 80% train  |  10% val  |  10% test  (stratified on diagnosis)
df_train, df_temp = train_test_split(
    df, test_size=0.2, stratify=df["diagnosis"], random_state=42
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5, stratify=df_temp["diagnosis"], random_state=42
)

print(f"Train  : {len(df_train):4d} samples")
print(f"Val    : {len(df_val):4d} samples")
print(f"Test   : {len(df_test):4d} samples")
print("\nTrain class distribution:")
print(df_train["diagnosis"].value_counts())

# Save splits for downstream notebooks
df_train.to_csv("../data/train_split.csv", index=False)
df_val.to_csv("../data/val_split.csv", index=False)
df_test.to_csv("../data/test_split.csv", index=False)
print("\nSplit CSV files saved.")


## 7. tf.data Pipeline Builders

In [ ]:
def build_tf_dataset(
    dataframe: pd.DataFrame,
    base_dir: Path,
    batch_size: int = BATCH_SIZE,
    augment: bool = False,
    shuffle: bool = True,
) -> tf.data.Dataset:
    """Create a tf.data.Dataset from a metadata dataframe.

    Each element is (image_tensor, label_int) where the image is
    loaded from disk, resized to 224x224, and normalised to [0, 1].

    Args:
        dataframe: DataFrame with columns image_path, diagnosis, label.
        base_dir: Root directory prepended to relative image_path values.
        batch_size: Mini-batch size.
        augment: Whether to apply random augmentations.
        shuffle: Whether to shuffle the dataset.

    Returns:
        Prefetched tf.data.Dataset ready for model training.
    """
    paths = [str(base_dir / p) for p in dataframe["image_path"]]
    labels = dataframe["label"].values

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        image = tf.image.decode_png(raw, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


# Test the pipeline
train_ds = build_tf_dataset(df_train, BASE_DIR, augment=True)
val_ds   = build_tf_dataset(df_val,   BASE_DIR, augment=False, shuffle=False)
test_ds  = build_tf_dataset(df_test,  BASE_DIR, augment=False, shuffle=False)

for images, labels in train_ds.take(1):
    print(f"Batch image shape : {images.shape}")
    print(f"Batch label shape : {labels.shape}")
    print(f"Pixel range       : [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")


## 8. Persist Preprocessing Objects

In [ ]:
import joblib

os.makedirs("../models/saved_models", exist_ok=True)
joblib.dump(le_gender, "../models/saved_models/le_gender.pkl")
joblib.dump(scaler_age, "../models/saved_models/scaler_age.pkl")
joblib.dump(le_label,  "../models/saved_models/le_label.pkl")
print("Preprocessing objects saved.")
